# Google Cloud Pub/Sub Hands-on Lab

## Pub/Sub Important Defaults / Limits

| Feature | Default | Min | Max |
|---|---:|---:|---:|
| Ack deadline | 10 sec | 10 sec | 600 sec |
| Message retention (subscription) | 7 days | 10 min | 31 days |
| Subscription expiration (inactive) | 31 days | 1 day | Never |
| Dead-letter max delivery attempts | 5 | 5 | 100 |
| Retry policy | Immediate retry | 0 sec | 600 sec backoff |
| Ordering | Disabled | - | Enabled per topic |
| Exactly-once delivery | Disabled | - | Enabled on pull subscriptions |
| Topic message retention | Disabled | 10 min | 31 days |
| Max message size | 10 MB | - | 10 MB |
| Max attributes | 100 | - | 100 |
| Schema validation | Disabled | - | Avro / Protocol Buffers |
| Push delivery | Disabled | - | HTTP/HTTPS endpoint |

> Notes:
> * Pub/Sub provides **at-least-once** delivery by default.
> * Messages are removed from a subscription after successful acknowledgement.
> * Dead-letter queues receive messages after the configured number of failed delivery attempts.
> * Every subscription maintains its own cursor; two subscriptions receive independent copies of the same message.


In [ ]:
PROJECT_ID="project-2df9d2ad-b6f5-4ed5-997"
REGION="us-east1"


#AUTH

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("authenticated")


In [ ]:
!gcloud config set project $PROJECT_ID

## Enable API

In [ ]:
!gcloud services enable pubsub.googleapis.com

## Create topic

In [ ]:
!gcloud pubsub topics create orders-topic

## Create subscription 1

In [ ]:
!gcloud pubsub subscriptions create orders-sub-1 --topic=orders-topic

## Create subscription 2

In [ ]:
!gcloud pubsub subscriptions create orders-sub-2 --topic=orders-topic

## Publish message

In [ ]:
!gcloud pubsub topics publish orders-topic --message='Order 1001 Created'

## Publish with attributes

In [ ]:
!gcloud pubsub topics publish orders-topic --message='Laptop Order' --attribute=customer=premium,priority=high

## Pull from sub1

In [ ]:
!gcloud pubsub subscriptions pull orders-sub-1 --limit=5

## Pull auto ack

In [ ]:
!gcloud pubsub subscriptions pull orders-sub-1 --limit=5 --auto-ack

## Manual ack

In [ ]:
# Pull first:
!gcloud pubsub subscriptions pull orders-sub-1 --limit=1 --format=json

# Copy ackId then:
# !gcloud pubsub subscriptions ack orders-sub-1 --ack-ids=ACK_ID

## Redelivery demo

In [ ]:
!gcloud pubsub subscriptions pull orders-sub-1 --limit=1
# Wait >10 seconds
!gcloud pubsub subscriptions pull orders-sub-1 --limit=1

## Create schema

In [ ]:
%%bash

cat > order.avsc <<EOF
{
  "type":"record",
  "name":"Order",
  "fields":[
    {"name":"orderId","type":"string"},
    {"name":"customer","type":"string"},
    {"name":"amount","type":"double"}
  ]
}
EOF
#

In [ ]:
!gcloud pubsub schemas create order-schema --type=avro --definition-file=order.avsc

## Schema topic

In [ ]:
!gcloud pubsub topics create schema-topic --schema=order-schema --message-encoding=json

## Publish valid schema message

In [ ]:
!gcloud pubsub topics publish schema-topic --message='{"orderId":"1001","customer":"Himanshu","amount":5500}'

## Publish invalid schema message

In [ ]:
!gcloud pubsub topics publish schema-topic --message='{"orderId":"1002","customer":"Himanshu"}'

## Dead Letter Queue

In [ ]:
!gcloud pubsub topics create orders-dlq
!gcloud pubsub subscriptions create orders-dlq-sub --topic=orders-dlq

PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = PROJECT_NUMBER[0]

print(PROJECT_NUMBER)

!gcloud beta services identity create --service=pubsub.googleapis.com --project={PROJECT_ID}

!gcloud pubsub topics add-iam-policy-binding orders-dlq \
 --member="serviceAccount:service-${PROJECT_NUMBER}@gcp-sa-pubsub.iam.gserviceaccount.com" \
 --role=roles/pubsub.publisher

!gcloud pubsub subscriptions create orders-sub-dlq \
 --topic=orders-topic \
 --dead-letter-topic=orders-dlq \
 --max-delivery-attempts=5

## Filtering

In [ ]:
!gcloud pubsub subscriptions create ordered-sub \
    --topic=orders-topic \
    --enable-message-ordering

## Ordering

In [ ]:
!gcloud pubsub topics update orders-topic --message-ordering
!gcloud pubsub topics publish orders-topic --message='Step 1' --ordering-key=customer1
!gcloud pubsub topics publish orders-topic --message='Step 2' --ordering-key=customer1

## Retention

In [ ]:
!gcloud pubsub subscriptions update orders-sub-1 --message-retention-duration=7d

## Snapshot

In [ ]:
!gcloud pubsub snapshots create orders-snapshot --subscription=orders-sub-1
!gcloud pubsub subscriptions seek orders-sub-1 --snapshot=orders-snapshot

## Exactly once

In [ ]:
!gcloud pubsub subscriptions create exactly-once-sub --topic=orders-topic --enable-exactly-once-delivery

## Cleanup

In [ ]:
!gcloud pubsub subscriptions delete orders-sub-1
!gcloud pubsub subscriptions delete orders-sub-2
!gcloud pubsub subscriptions delete orders-sub-dlq
!gcloud pubsub subscriptions delete orders-dlq-sub
!gcloud pubsub subscriptions delete premium-orders
!gcloud pubsub subscriptions delete exactly-once-sub
!gcloud pubsub snapshots delete orders-snapshot
!gcloud pubsub topics delete orders-topic
!gcloud pubsub topics delete schema-topic
!gcloud pubsub topics delete orders-dlq
!gcloud pubsub schemas delete order-schema